# Retrain the oil palm crown detector on two sites at once (West Africa + Peru)

Fine-tuning the released detector on Peru photos alone raised the share of Peru palms it finds from 47% to
97%, but it then failed on the West African plantation it was built for (test mAP@0.5 fell from 0.844 to
0.004). This notebook trains one model on both sites together and checks both.

1. Downloads the released model (run 9) from Hugging Face and the public Peru dataset from Mendeley Data,
   and rebuilds exactly the Peru split used before (neighbouring photos kept together).
2. Reads the labelled West African tiles from a **private** Kaggle dataset and checks they are complete.
3. Shrinks the Peru photos so their palms appear at the same size as the West African palms, and packs
   them 16 to a 1280 px picture, the size the West African tiles are trained at.
4. Retrains from the released model on both sites with two learning rates and keeps the one with the best
   validation score averaged over the two sites.
5. Scores the released and retrained models once on both test sets and saves `joint_finetune_results.zip`.

**Privacy.** The West African tiles are the plantation owner's imagery, used here with their permission.
Keep this notebook and the tiles dataset **private**. The tiles, the training runs and every picture made
from the tiles stay in a scratch folder that Kaggle does not save; only scores, counts, curves, the model
and pictures of the public Peru photos go into the saved output.

**Settings.** Kaggle: *Add Input* → *Your Datasets* → the private tiles dataset; *Session options* →
*Accelerator* *GPU T4 x2* (one GPU is used) and *Internet* on. The published run took about an hour on a T4 (21 and 35 minutes of training for the
two learning rates).

**Data.** Peru: *Oil Palm Tree Detection for Anomaly Identification*, Dominguez Meza & Rituay, Mendeley
Data, [doi:10.17632/nh7d23dgnw.1](https://doi.org/10.17632/nh7d23dgnw.1), CC BY 4.0 (healthy and anomalous
palms counted as one class). Code: [github.com/Dexsholex/oil-palm-crown-detection](https://github.com/Dexsholex/oil-palm-crown-detection)
(AGPL-3.0); model DOI [10.5281/zenodo.23704654](https://doi.org/10.5281/zenodo.23704654).

In [ ]:
# The Ultralytics version the released model was checked with.
!pip install -q ultralytics==8.4.178

In [ ]:
import csv, hashlib, io, json, os, random, re, shutil, time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import torch
import ultralytics
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

SMOKE = os.environ.get("PALM_SMOKE") == "1"  # tiny CPU run used to test this notebook; leave unset

# Kaggle saves /kaggle/working as the notebook's output. Anything made from the plantation tiles goes to TEMP,
# which Kaggle does not save.
WORK = Path(os.environ.get("PALM_WORK", "/kaggle/working" if Path("/kaggle/working").is_dir() else "joint_work"))
TEMP = Path(os.environ.get("PALM_TEMP", "/kaggle/temp" if Path("/kaggle").is_dir() else WORK / "temp"))
RESULTS = WORK / "results"

WEIGHTS_URL = "https://huggingface.co/dexsholex/oil-palm-crown-detector/resolve/main/oil_palm_crown.pt"
WEIGHTS_MD5 = "b219fd1f5a44575061196bfe0f9635dd"
DATA_URL = ("https://data.mendeley.com/public-files/datasets/nh7d23dgnw/files/"
            "c4cba156-35e5-415a-a8bc-9054c5e41eda/file_downloaded")
DATA_MD5 = "54070f603055e8166295a4ffbba2c55a"

# The West African tiles as published in the model card: (tiles, labelled crowns) per split.
WA_EXPECTED = {"train": (313, 2898), "val": (43, 427), "test": (79, 716)}
# The Peru split from the Peru-only notebook: (photos, palms) per split.
PERU_EXPECTED = {"train": (276, 354), "val": (64, 80), "test": (84, 115)}

SEED = 0
BLOCK = 20                        # Peru photos numbered in the same block of 20 go to the same split
TEST_SHARE, VAL_SHARE = 0.15, 0.15
NEAR_DUPLICATE = 0.5
PERU_CELL = (320, 240)            # 800 x 600 photos shrunk 0.4x: palms ~196 px, like the West African palms
PERU_GRID = (4, 4)                # 16 photos per 1280 x 960 training picture
PERU_PACKINGS = 6                 # six different shuffles, so the Peru palms weigh about as much as the tiles
LEARNING_RATES = [0.0002, 0.001]  # run 9's rate and a 5x higher one
BASELINE_SIZES = [256, 320, 480, 640]
WA_IMGSZ, PERU_IMGSZ = 1280, 320  # tiles at their native size; single Peru photos at the packed scale
EPOCHS, PATIENCE, BATCH, WORKERS, FRACTION = 150, 30, 8, 2, 1.0  # run 9: 150 epochs, patience 30
CONF, NMS_IOU, INTERIOR_MARGIN = 0.25, 0.45, 128  # pipeline settings; margin = half the 256 px tile overlap
DEVICE = 0
# Run 9's published scores on the West African test tiles, used to check this notebook's scoring.
PUBLISHED_RUN9 = {"precision": 0.845, "recall": 0.774, "mAP50": 0.844, "interior_model": 519, "interior_manual": 518}

if SMOKE:
    LEARNING_RATES, BASELINE_SIZES, PERU_PACKINGS = [0.001], [256], 1
    EPOCHS, PATIENCE, BATCH, WORKERS, FRACTION, DEVICE = 1, 1, 2, 0, 1.0, "cpu"
elif not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Kaggle: Session options → Accelerator → GPU T4 x2.")

for folder in [RESULTS, TEMP]:
    if folder.exists():
        shutil.rmtree(folder)  # start clean when the notebook is re-run
    folder.mkdir(parents=True)
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("Saved output:", RESULTS.resolve(), "| scratch (not saved):", TEMP.resolve(), "| GPU:", GPU)

In [ ]:
def fetch(url, dest, md5):
    """Download once, then check the file is exactly the one this notebook was written for."""
    if not dest.exists():
        print("Downloading", dest.name, "...")
        part = dest.with_suffix(".part")
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request) as response, open(part, "wb") as out:
            shutil.copyfileobj(response, out)
        part.rename(dest)
    digest = hashlib.md5(dest.read_bytes()).hexdigest()
    if digest != md5:
        raise RuntimeError(f"{dest.name} has checksum {digest}, expected {md5}. Delete it and re-run.")
    return dest


weights_run9 = fetch(WEIGHTS_URL, Path(os.environ.get("PALM_WEIGHTS", TEMP / "oil_palm_crown_run9.pt")), WEIGHTS_MD5)
peru_zip = fetch(DATA_URL, Path(os.environ.get("PALM_PERU_ZIP", TEMP / "peru_oil_palm.zip")), DATA_MD5)

archive = zipfile.ZipFile(peru_zip)
labels, size = {}, {}  # photo path in the zip -> palm boxes (x_min, y_min, x_max, y_max) and (width, height)
for part in ["train", "valid", "test"]:
    with archive.open(f"{part}/_annotations.csv") as fh:
        for row in csv.DictReader(io.TextIOWrapper(fh, encoding="utf-8")):
            path = f"{part}/{row['filename']}"
            labels.setdefault(path, []).append(tuple(float(row[k]) for k in ("xmin", "ymin", "xmax", "ymax")))
            size[path] = (int(row["width"]), int(row["height"]))
photos = sorted(n for n in archive.namelist() if n.endswith(".jpg"))
for p in photos:
    if p not in size:  # a photo with no palms has no row in the label file
        size[p] = Image.open(archive.open(p)).size


def photo_number(path):
    return int(re.search(r"imgPalm(\d+)_", path).group(1))

## The Peru split

The same split as the Peru-only notebook: blocks of 20 consecutive photo numbers go to one split, and any
validation or test photo that looks almost the same as a training photo moves to training. The counts are
checked against that run, so both notebooks are scored on the same test photos.

In [ ]:
blocks = sorted({photo_number(p) // BLOCK for p in photos})
random.Random(SEED).shuffle(blocks)
split_of_block, filled = {}, {"test": 0, "val": 0, "train": 0}
for b in blocks:
    target = ("test" if filled["test"] < TEST_SHARE * len(photos)
              else "val" if filled["val"] < VAL_SHARE * len(photos) else "train")
    split_of_block[b] = target
    filled[target] += sum(1 for p in photos if photo_number(p) // BLOCK == b)
split = {p: split_of_block[photo_number(p) // BLOCK] for p in photos}


def thumbnail(path):
    """Small grey thumbnail, scaled so that the dot product of two thumbnails is their correlation."""
    grey = np.asarray(Image.open(archive.open(path)).convert("L").resize((64, 48), Image.BILINEAR), np.float32)
    grey = (grey - grey.mean()) / (grey.std() + 1e-6)
    return grey.ravel() / np.sqrt(grey.size)


similarity = np.stack([thumbnail(p) for p in photos])
similarity = similarity @ similarity.T


def near_duplicates(assignment):
    in_train = np.array([assignment[p] == "train" for p in photos])
    return {p: float(similarity[i, in_train].max()) for i, p in enumerate(photos)
            if not in_train[i] and similarity[i, in_train].max() >= NEAR_DUPLICATE}


while leaks := near_duplicates(split):
    for p in leaks:
        split[p] = "train"

peru_split = {s: (sum(split[p] == s for p in photos), sum(len(labels.get(p, [])) for p in photos if split[p] == s))
              for s in ["train", "val", "test"]}
if peru_split != PERU_EXPECTED:
    raise RuntimeError(f"Peru split {peru_split} differs from the Peru-only run {PERU_EXPECTED}.")
print("Peru split (photos, palms):", peru_split)

# One photo per file, for scoring at PERU_IMGSZ.
PERU = TEMP / "peru_yolo"
for p in photos:
    folder = PERU / split[p]
    (folder / "images").mkdir(parents=True, exist_ok=True)
    (folder / "labels").mkdir(parents=True, exist_ok=True)
    stem, (w, h) = Path(p).stem, size[p]
    (folder / "images" / f"{stem}.jpg").write_bytes(archive.read(p))
    lines = [f"0 {(x0 + x1) / 2 / w:.6f} {(y0 + y1) / 2 / h:.6f} {(x1 - x0) / w:.6f} {(y1 - y0) / h:.6f}"
             for x0, y0, x1, y1 in labels.get(p, [])]
    (folder / "labels" / f"{stem}.txt").write_text("\n".join(lines))
PERU_YAML = PERU / "data.yaml"
PERU_YAML.write_text(f"path: {PERU.resolve().as_posix()}\ntrain: train/images\nval: val/images\n"
                     "test: test/images\nnames:\n  0: oil_palm\n")

## The West African tiles (private)

Found in the attached private dataset, checked against the published counts, and copied to the scratch
folder that Kaggle does not save.

In [ ]:
def find_tiles():
    if os.environ.get("PALM_WA_DIR"):
        return Path(os.environ["PALM_WA_DIR"])
    for images in sorted(Path("/kaggle/input").glob("**/train/images")):
        if any(f.name.startswith(("pilot_", "block_")) for f in images.iterdir()):
            return images.parent.parent
    raise RuntimeError("West African tiles not found. Right-hand panel: Add Input → Your Datasets → the tiles.")


source = find_tiles()
WA = TEMP / "wa"
for part, (n_tiles, n_boxes) in WA_EXPECTED.items():
    tiles = sorted((source / part / "images").glob("*.jpg"))
    boxes = sum(sum(1 for line in (source / part / "labels" / f"{t.stem}.txt").read_text().splitlines()
                    if line.strip()) for t in tiles)
    if (len(tiles), boxes) != (n_tiles, n_boxes):
        raise RuntimeError(f"{part}: {len(tiles)} tiles and {boxes} crowns, expected {n_tiles} and {n_boxes}.")
    if SMOKE:
        tiles = tiles[:4]
    for kind in ["images", "labels"]:
        (WA / part / kind).mkdir(parents=True)
    for t in tiles:
        shutil.copy(t, WA / part / "images" / t.name)
        shutil.copy(source / part / "labels" / f"{t.stem}.txt", WA / part / "labels" / f"{t.stem}.txt")
WA_YAML = WA / "data.yaml"
WA_YAML.write_text(f"path: {WA.resolve().as_posix()}\ntrain: train/images\nval: val/images\n"
                   "test: test/images\nnames:\n  0: oil_palm\n")
print("West African tiles checked:", {s: v for s, v in WA_EXPECTED.items()})

## Peru photos at the West African palm size

YOLO resizes every training picture to 1280 px on its longest side. A single 800 × 600 Peru photo would be
blown up and its palms would appear about 790 px wide, four times the West African palms. So each photo is
shrunk to 320 × 240 (palms about 196 px) and 16 are packed into one 1280 × 960 picture, with their boxes
moved to match. Six different shuffles give the Peru palms roughly the same weight in training as the
West African crowns.

In [ ]:
def pack(part, packings):
    folder = TEMP / "peru_packed" / part
    for kind in ["images", "labels"]:
        (folder / kind).mkdir(parents=True, exist_ok=True)
    cols, rows = PERU_GRID
    cw, ch = PERU_CELL
    names = [p for p in photos if split[p] == part]
    count = 0
    for k in range(packings):
        order = list(names)
        random.Random(SEED + k).shuffle(order)
        for start in range(0, len(order), cols * rows):
            canvas = Image.new("RGB", (cols * cw, rows * ch), (114, 114, 114))
            lines = []
            for i, p in enumerate(order[start:start + cols * rows]):
                w, h = size[p]
                ox, oy, s = (i % cols) * cw, (i // cols) * ch, cw / w
                canvas.paste(Image.open(archive.open(p)).convert("RGB").resize((cw, round(h * s)), Image.BILINEAR), (ox, oy))
                for x0, y0, x1, y1 in labels.get(p, []):
                    lines.append(f"0 {(ox + (x0 + x1) / 2 * s) / canvas.width:.6f} {(oy + (y0 + y1) / 2 * s) / canvas.height:.6f} "
                                 f"{(x1 - x0) * s / canvas.width:.6f} {(y1 - y0) * s / canvas.height:.6f}")
            name = f"peru_packed_{k}_{start // (cols * rows):02d}"
            canvas.save(folder / "images" / f"{name}.jpg", quality=95)
            (folder / "labels" / f"{name}.txt").write_text("\n".join(lines))
            count += 1
    return count


assert all(size[p] == (800, 600) for p in photos), "packing assumes 800 x 600 photos"
packed = {"train": pack("train", PERU_PACKINGS), "val": pack("val", 1)}
JOINT_YAML = TEMP / "joint.yaml"
JOINT_YAML.write_text(f"path: {TEMP.resolve().as_posix()}\n"
                      "train: [wa/train/images, peru_packed/train/images]\n"
                      "val: [wa/val/images, peru_packed/val/images]\n"
                      "names:\n  0: oil_palm\n")
print("Packed Peru training pictures:", packed)

In [ ]:
def score(weights, data_yaml, part, imgsz, name):
    """Ultralytics' standard detection scores on one split (precision and recall at the F1-best confidence)."""
    m = YOLO(str(weights)).val(data=str(data_yaml), split=part, imgsz=imgsz, batch=BATCH, device=DEVICE,
                               workers=WORKERS, plots=False, verbose=False,
                               project=str(TEMP / "val"), name=name, exist_ok=True)
    return {"precision": m.box.mp, "recall": m.box.mr, "mAP50": m.box.map50, "mAP50_95": m.box.map}


def iou_matrix(a, b):
    x0, y0 = np.maximum(a[:, None, 0], b[None, :, 0]), np.maximum(a[:, None, 1], b[None, :, 1])
    x1, y1 = np.minimum(a[:, None, 2], b[None, :, 2]), np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x1 - x0, 0, None) * np.clip(y1 - y0, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def count_matches(pred, conf, gt, thresh=0.5):
    """Greedy one-to-one matching, highest-confidence detection first (as in scripts/evaluate_test_split.py)."""
    if len(pred) == 0 or len(gt) == 0:
        return 0
    used, matched = set(), 0
    for row in iou_matrix(pred[np.argsort(-conf)], gt):
        for j in np.argsort(-row):
            if row[j] < thresh:
                break
            if j not in used:
                used.add(j)
                matched += 1
                break
    return matched


def predict(model, image, label_file, imgsz):
    r = model.predict(str(image), imgsz=imgsz, conf=CONF, iou=NMS_IOU, device=DEVICE, verbose=False)[0]
    pred, conf = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
    h, w = r.orig_shape
    lab = np.loadtxt(label_file, ndmin=2)
    gt = (np.stack([(lab[:, 1] - lab[:, 3] / 2) * w, (lab[:, 2] - lab[:, 4] / 2) * h,
                    (lab[:, 1] + lab[:, 3] / 2) * w, (lab[:, 2] + lab[:, 4] / 2) * h], axis=1)
          if lab.size else np.zeros((0, 4)))
    return pred, conf, gt, w, h


def wa_counts(weights, part="test"):
    """Tile counts at the pipeline's settings, as scripts/evaluate_test_split.py computes them."""
    model, rows = YOLO(str(weights)), []

    def interior(boxes, w, h):
        cx, cy = (boxes[:, 0] + boxes[:, 2]) / 2, (boxes[:, 1] + boxes[:, 3]) / 2
        m = INTERIOR_MARGIN
        return int(((cx >= m) & (cx < w - m) & (cy >= m) & (cy < h - m)).sum())

    for img in sorted((WA / part / "images").glob("*.jpg")):
        pred, conf, gt, w, h = predict(model, img, WA / part / "labels" / f"{img.stem}.txt", WA_IMGSZ)
        rows.append({"tile": img.stem, "block": img.stem.rsplit("_r", 1)[0], "manual_count": len(gt),
                     "model_count": len(pred), "matched": count_matches(pred, conf, gt),
                     "manual_interior": interior(gt, w, h), "model_interior": interior(pred, w, h)})
    total = lambda key, block=None: sum(r[key] for r in rows if block in (None, r["block"]))
    out = {"tiles": len(rows),
           "precision_iou50": total("matched") / max(total("model_count"), 1),
           "recall_iou50": total("matched") / max(total("manual_count"), 1),
           "whole_tile": f"{total('model_count')} vs {total('manual_count')}",
           "interior": f"{total('model_interior')} vs {total('manual_interior')}",
           "interior_error_pct": 100 * (total("model_interior") - total("manual_interior")) / max(total("manual_interior"), 1),
           "interior_mae_per_tile": float(np.mean([abs(r["model_interior"] - r["manual_interior"]) for r in rows]))}
    for block in sorted({r["block"] for r in rows}):
        out[f"interior_{block}"] = f"{total('model_interior', block)} vs {total('manual_interior', block)}"
    return out, rows


def peru_counts(weights, imgsz, part="test"):
    """Counts photo by photo; a palm is 'found' when a detection is centred inside its label."""
    model = YOLO(str(weights))
    t = dict(labelled=0, detected=0, found=0, on_palm=0, empty=0)
    boxes = {}
    for img in sorted((PERU / part / "images").glob("*.jpg")):
        pred, conf, gt, w, h = predict(model, img, PERU / part / "labels" / f"{img.stem}.txt", imgsz)
        cx, cy = (pred[:, 0] + pred[:, 2]) / 2, (pred[:, 1] + pred[:, 3]) / 2
        inside = ((cx[:, None] >= gt[None, :, 0]) & (cx[:, None] <= gt[None, :, 2]) &
                  (cy[:, None] >= gt[None, :, 1]) & (cy[:, None] <= gt[None, :, 3]))
        t["labelled"] += len(gt)
        t["detected"] += len(pred)
        t["found"] += int(inside.any(axis=0).sum())
        t["on_palm"] += int(inside.any(axis=1).sum())
        t["empty"] += int(len(pred) == 0)
        boxes[img.stem] = pred
    return {"photos": len(boxes), "palms_found": t["found"] / max(t["labelled"], 1),
            "detections_on_a_palm": t["on_palm"] / max(t["detected"], 1),
            "count": f"{t['detected']} vs {t['labelled']}",
            "count_error_pct": 100 * (t["detected"] - t["labelled"]) / max(t["labelled"], 1),
            "photos_without_detection": t["empty"]}, boxes

## Before: the released model

Its Peru input size is chosen on the Peru validation photos, as in the Peru-only notebook. Its scores on
the West African test tiles should match the published ones; this checks the scoring code in this notebook.

In [ ]:
baseline_val = {s: score(weights_run9, PERU_YAML, "val", s, f"run9_peru_val_{s}") for s in BASELINE_SIZES}
baseline_size = max(BASELINE_SIZES, key=lambda s: baseline_val[s]["mAP50"])
print("Released model's best Peru input size:", baseline_size)

## Training on both sites

Starts from the released model with run 9's recipe (AdamW, rotations up to 180°, flips, mosaic, 1280 px).
Each learning rate is scored on both validation sets, and the one with the best average mAP@0.5 is kept.
Training runs and their sample pictures stay in the scratch folder; only the curves are saved.

In [ ]:
trials = []
for lr in LEARNING_RATES:
    name = f"joint_lr{lr:g}"
    start = time.time()
    YOLO(str(weights_run9)).train(
        data=str(JOINT_YAML), imgsz=WA_IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, fraction=FRACTION,
        optimizer="AdamW", lr0=lr, lrf=0.01, momentum=0.937, weight_decay=0.0005, warmup_epochs=2.0,
        degrees=180, translate=0.1, scale=0.5, shear=0.0, perspective=0.0, flipud=0.5, fliplr=0.5,
        mosaic=1.0, mixup=0.0, close_mosaic=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        cache="ram", seed=SEED, deterministic=True, device=DEVICE, workers=WORKERS,
        plots=not SMOKE, project=str(TEMP / "train"), name=name, exist_ok=True)
    best = TEMP / "train" / name / "weights" / "best.pt"
    trial = {"name": name, "lr0": lr, "weights": best, "minutes": round((time.time() - start) / 60, 1),
             "wa_val": score(best, WA_YAML, "val", WA_IMGSZ, f"{name}_wa_val"),
             "peru_val": score(best, PERU_YAML, "val", PERU_IMGSZ, f"{name}_peru_val")}
    trial["mean_val_mAP50"] = (trial["wa_val"]["mAP50"] + trial["peru_val"]["mAP50"]) / 2
    trials.append(trial)
    # Save the curves only: the sample pictures Ultralytics draws show plantation tiles.
    curves = RESULTS / "train" / name
    curves.mkdir(parents=True)
    for f in (TEMP / "train" / name).iterdir():
        if f.name in ("results.csv", "results.png", "args.yaml") or f.name.endswith("_curve.png"):
            shutil.copy(f, curves / f.name)
    print(f"{name}: {trial['minutes']} min, validation mAP50 West Africa {trial['wa_val']['mAP50']:.3f}, "
          f"Peru {trial['peru_val']['mAP50']:.3f}")
chosen = max(trials, key=lambda t: t["mean_val_mAP50"])
FINAL = RESULTS / "oil_palm_crown_joint.pt"
shutil.copy(chosen["weights"], FINAL)
print("Kept", chosen["name"])

## After: one look at both test sets

In [ ]:
models = [("Released model (run 9)", weights_run9, baseline_size), ("Trained on both sites", FINAL, PERU_IMGSZ)]
test, peru_boxes = [], {}
for label, weights, peru_size in models:
    wa, wa_rows = wa_counts(weights)
    tag = "run9" if weights == weights_run9 else "joint"
    with open(RESULTS / f"wa_test_counts_{tag}.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(wa_rows[0]))
        writer.writeheader()
        writer.writerows(wa_rows)
    peru, peru_boxes[label] = peru_counts(weights, peru_size)
    test.append({"model": label,
                 "west_africa": {**score(weights, WA_YAML, "test", WA_IMGSZ, f"{tag}_wa_test"), **wa},
                 "peru": {"imgsz": peru_size, **score(weights, PERU_YAML, "test", peru_size, f"{tag}_peru_test"), **peru}})

print("| Model | WA precision | WA recall | WA mAP@0.5 | WA interior count | Peru mAP@0.5 | Peru palms found | Peru count |")
print("|---|---|---|---|---|---|---|---|")
for r in test:
    wa, pe = r["west_africa"], r["peru"]
    print(f"| {r['model']} | {wa['precision']:.3f} | {wa['recall']:.3f} | {wa['mAP50']:.3f} | {wa['interior']} "
          f"({wa['interior_error_pct']:+.1f}%) | {pe['mAP50']:.3f} | {100 * pe['palms_found']:.1f}% | "
          f"{pe['count']} ({pe['count_error_pct']:+.1f}%) |")

released = test[0]["west_africa"]
check = {"precision": round(released["precision"], 3), "recall": round(released["recall"], 3),
         "mAP50": round(released["mAP50"], 3), "interior": released["interior"]}
expected = {"precision": PUBLISHED_RUN9["precision"], "recall": PUBLISHED_RUN9["recall"],
            "mAP50": PUBLISHED_RUN9["mAP50"],
            "interior": f"{PUBLISHED_RUN9['interior_model']} vs {PUBLISHED_RUN9['interior_manual']}"}
print("Released model vs its published West African scores:", "match" if check == expected else f"{check} vs {expected}")

## Pictures and saved output

Pictures use the public Peru test photos only. Each shows the dataset's hand labels, the released model and
the model trained on both sites, with the palm count over each panel.

In [ ]:
try:
    font = ImageFont.load_default(size=22)  # Pillow 10.1+ with FreeType
    centred = {"anchor": "mm"}
except TypeError:
    font, centred = ImageFont.load_default(), {}
by_stem = {Path(p).stem: p for p in photos}
examples = sorted(peru_boxes[models[0][0]], key=lambda s: (-len(labels.get(by_stem[s], [])), s))[:4]
(RESULTS / "examples").mkdir()
for stem in examples:
    panels = [("Hand labels (dataset)", np.array(labels.get(by_stem[stem], [])).reshape(-1, 4), "#00e676")]
    panels += [(label, peru_boxes[label][stem], "#ffd600") for label, _, _ in models]
    framed = []
    for title, boxes, colour in panels:
        im = Image.open(archive.open(by_stem[stem])).convert("RGB")
        draw = ImageDraw.Draw(im)
        for x0, y0, x1, y1 in boxes:
            draw.rectangle([x0, y0, x1, y1], outline=colour, width=6)
        im = im.resize((im.width // 2, im.height // 2))
        panel = Image.new("RGB", (im.width, im.height + 40), "white")
        panel.paste(im, (0, 40))
        n = len(boxes)
        ImageDraw.Draw(panel).text((im.width // 2, 20) if centred else (10, 10),
                                   f"{title}: {n} palm{'' if n == 1 else 's'}", fill="black", font=font, **centred)
        framed.append(panel)
    sheet = Image.new("RGB", (sum(p.width for p in framed) + 20, framed[0].height), "white")
    for i, panel in enumerate(framed):
        sheet.paste(panel, (i * (panel.width + 10), 0))
    sheet.save(RESULTS / "examples" / f"{stem}.jpg", quality=90)

summary = {
    "settings": {"seed": SEED, "learning_rates": LEARNING_RATES, "epochs": EPOCHS, "patience": PATIENCE,
                 "batch": BATCH, "wa_imgsz": WA_IMGSZ, "peru_imgsz": PERU_IMGSZ, "peru_cell": PERU_CELL,
                 "peru_grid": PERU_GRID, "peru_packings": PERU_PACKINGS, "conf": CONF, "nms_iou": NMS_IOU,
                 "interior_margin": INTERIOR_MARGIN, "smoke_test": SMOKE},
    "peru_split": peru_split, "wa_split": WA_EXPECTED, "packed_peru_pictures": packed,
    "released_peru_validation": baseline_val, "released_peru_size": baseline_size,
    "trials": [{k: v for k, v in t.items() if k != "weights"} for t in trials],
    "chosen": chosen["name"], "test": test,
    "released_matches_published": check == expected,
    "versions": {"ultralytics": ultralytics.__version__, "torch": torch.__version__, "gpu": GPU},
}
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2, default=float))

bundle = WORK / "joint_finetune_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as out:
    for f in sorted(RESULTS.rglob("*")):
        if f.is_file():
            out.write(f, f.relative_to(RESULTS))
print(f"Saved {bundle} ({bundle.stat().st_size / 1e6:.0f} MB)")

## Next

Download `joint_finetune_results.zip` from the *Output* panel. It holds the model trained on both sites
(`oil_palm_crown_joint.pt`), `summary.json` with every score above, per-tile West African counts, training
curves and the Peru pictures. The results of the published run are in the repository's `evaluation/`
folder. Then delete the private tiles dataset from Kaggle if you agreed to with the plantation owner.